# Is skipping null steps correct?

`skip_null_steps=True` only simulates the steps that change a node and draws how many null steps were skipped in between. If that is correct, the output has the same distribution as the plain engine.

Below, both engines run on a few small graphs. For each run we split `steps` into steps with a change (`n_changes`) and null steps (`steps - n_changes`), and compare their distributions between the engines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

from moran_graphs import PopulationGraph as pg, simulate

graphs = {
    "complete N=20": pg.complete_graph(20),
    "star N=20": pg.star_graph(20),
    "mammalian lung N=31": pg.mammalian_lung_graph(depth=4),
    "random N=31": pg.random_graph(31, 40, seed=1),
}
colors = {"plain": "#2a78d6", "skip": "#eb6834"}


def run_both(graphs, r, n_repeats):
    return {
        name: {
            "plain": simulate(g.csr(), [1], [r], n_repeats, 10**10, seed=0),
            "skip": simulate(g.csr(), [1], [r], n_repeats, 10**10, seed=1, skip_null_steps=True),
        }
        for name, g in graphs.items()
    }


runs = run_both(graphs, r=1.5, n_repeats=20_000)

Means per run. Time is conditional on fixation. The p-values are two-sample Kolmogorov-Smirnov tests on the fixed runs; a large p-value means no detectable difference.

In [ ]:
def counts(res):
    fixed = res["winner"] == 0
    return {"changes": res["n_changes"][fixed], "null steps": res["steps"][fixed] - res["n_changes"][fixed]}


def summary_table(runs):
    rows = []
    for name, pair in runs.items():
        plain, skip = counts(pair["plain"]), counts(pair["skip"])
        row = {"graph": name,
               "rho plain": (pair["plain"]["winner"] == 0).mean(), "rho skip": (pair["skip"]["winner"] == 0).mean()}
        for column in ["changes", "null steps"]:
            row[f"{column} plain"] = plain[column].mean()
            row[f"{column} skip"] = skip[column].mean()
            row[f"{column} KS p"] = ks_2samp(plain[column], skip[column]).pvalue
        rows.append(row)
    return pd.DataFrame(rows).round(3)


summary_table(runs)

## Reading a CDF

The plots below show cumulative distribution functions (CDFs). For a quantity such as "null steps in a run", the CDF at a value $x$ is the fraction of runs with at most $x$ null steps. The curve starts at 0 on the left, climbs to 1 on the right, and rises steeply where many runs land. Where it crosses 0.5 is the median.

Each curve here is the empirical CDF: the values of all fixed runs, sorted, with the $i$-th smallest plotted at height $i/n$. Unlike a mean, it shows the whole distribution (shape, spread, tail), and unlike a histogram it needs no choice of bins. The KS p-value in the table above is based on the largest vertical gap between the two curves.

If skipping null steps is correct, the plain and skip curves lie on top of each other, up to small wiggles from random noise.

In [ ]:
def plot_cdfs(runs):
    fig, axes = plt.subplots(len(runs), 2, figsize=(11, 3 * len(runs)))
    for row_axes, (name, pair) in zip(axes, runs.items()):
        for ax, column in zip(row_axes, ["changes", "null steps"]):
            for engine, res in pair.items():
                x = np.sort(counts(res)[column])
                ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=colors[engine], lw=1.5, label=engine)
            ax.set(xscale="log", xlabel=column, ylabel="CDF", title=f"{name}: {column}")
            ax.spines[["top", "right"]].set_visible(False)
            ax.legend()
    plt.tight_layout()


plot_cdfs(runs)

## More graphs, slightly bigger, r = 1.1

Graphs of about 60 nodes, including the respiratory topologies. At r = 1.1 the process is close to neutral, so runs are long and most steps are null: this is where skipping matters most.

In [ ]:
bigger_graphs = {
    "complete N=63": pg.complete_graph(63),
    "star N=63": pg.star_graph(63),
    "mammalian lung N=63": pg.mammalian_lung_graph(depth=5),
    "asymmetric lung N=63": pg.asymmetric_lung_graph(n_nodes=63),
    "avian N=63": pg.avian_graph(4, 15),
    "fish N=64": pg.fish_graph(4, 5),
    "random N=63": pg.random_graph(63, 90, seed=1),
}
bigger_runs = run_both(bigger_graphs, r=1.1, n_repeats=5_000)
display(summary_table(bigger_runs))
plot_cdfs(bigger_runs)